# CYA Colab setup

Run this notebook through the official VS Code Colab extension. It mounts Drive, prepares the disposable runtime checkout, installs dependencies without replacing Colab's PyTorch/CUDA build, optionally stages data under `/content`, and runs the strict smoke check.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_FOLDER_ID = "1c-IVvAiHlApA49CtU3QQH9XqQDmkbO8U"
DRIVE_FOLDER_URL = f"https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}"
DRIVE_DATA_ROOT = DRIVE_ROOT / "hackathon_data"
DRIVE_ARTIFACT_ROOT = DRIVE_ROOT / "cya-techjam26/artifacts"
LOCAL_PROJECT_ROOT = Path("/content/cya-techjam26")
LOCAL_DATA_ROOT = Path("/content/hackathon_data")
REPOSITORY_URL = "https://github.com/maxi-cmyk/cya-techjam26.git"

assert DRIVE_DATA_ROOT.exists(), (
    f"Drive dataset not found: {DRIVE_DATA_ROOT}. "
    f"Open {DRIVE_FOLDER_URL}, add a shortcut to My Drive, and name it hackathon_data."
)
DRIVE_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Drive folder ID: {DRIVE_FOLDER_ID}")
print(f"Drive data: {DRIVE_DATA_ROOT}")
print(f"Persistent artifacts: {DRIVE_ARTIFACT_ROOT}")

In [ ]:
import subprocess

if (LOCAL_PROJECT_ROOT / ".git").exists():
    subprocess.run(
        ["git", "pull", "--ff-only"],
        cwd=LOCAL_PROJECT_ROOT,
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(LOCAL_PROJECT_ROOT)],
        check=True,
    )

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"],
    cwd=LOCAL_PROJECT_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"],
    cwd=LOCAL_PROJECT_ROOT,
    check=True,
)

## Optional local data staging

Keep `COPY_DATA = False` until `DATA_SUBDIRECTORY` matches the Drive folder needed for the current task. Copying can take time, but training from `/content` avoids repeated small-file reads through the Drive mount.

In [ ]:
import shutil

COPY_DATA = False
DATA_SUBDIRECTORY = Path("cleaned/sid_set")  # Change to raw/sid_set for Task 2.

source_data = DRIVE_DATA_ROOT / DATA_SUBDIRECTORY
local_data = LOCAL_DATA_ROOT / DATA_SUBDIRECTORY
assert source_data.exists(), f"Dataset subset not found: {source_data}"

if COPY_DATA:
    shutil.copytree(source_data, local_data, dirs_exist_ok=True)
    print(f"Staged data at {local_data}")
else:
    print(f"Ready to copy {source_data} to {local_data}")

In [ ]:
subprocess.run(
    [sys.executable, "scripts/smoke_check.py", "--config", "configs/colab.json"],
    cwd=LOCAL_PROJECT_ROOT,
    check=True,
)

## Sync durable outputs

Run this after a completed experiment. It copies rather than moves, so an interrupted Drive operation does not remove the local output.

In [ ]:
local_artifacts = LOCAL_PROJECT_ROOT / "artifacts"
if local_artifacts.exists():
    shutil.copytree(local_artifacts, DRIVE_ARTIFACT_ROOT, dirs_exist_ok=True)
    print(f"Copied artifacts to {DRIVE_ARTIFACT_ROOT}")
else:
    print("No local artifacts to sync yet.")